# Crossing Watch: our tuned Glimmer judge (LoRA v6) on Kaggle, for $0

Our LoRA-tuned Muse Glimmer judge, v6 (`accounts/<fireworks-account>/models/muse-starter-sl-judge-v6-main`: rank 32,
alpha 32, trained on Fireworks' Serverless Training API in session `ts-17bf2de281bb4cbe8424de262befb01b`), runs here
on Kaggle's free **2 x T4**: Meta's official 4-bit Glimmer base (sha256 checked), the same llama.cpp `b11976` as
`kaggle/glimmer-gate.ipynb`, and our adapter, converted to GGUF in this notebook. No API of any kind is called.

At temperature 0, as the training session scored it, the judge reads:
1. the public planted-lies bench (`kaggle/judge-bench-public.json`) through the product's own judge path (`judge_bench.py`);
2. our real gate letter, 3 plain reads and 3 with the product's JSON schema, on the prompt the Fireworks deployment
   read on 3 Nov 2026 (sha256 `3d9bf4a1…`) and on today's product prompt;
3. all 1,545 v6 holdout prompts, in holdout order, as `provenance/glimmer-judge-lora-v6/live/replay_v6.py` sent them
   to Fireworks (one user message, 256 tokens).

**Pass bar, written before this run (Wed 4 Nov 2026, 05:40 IST; the same text is in GATES.md).** Scored on v6's 1,501
test rows with the labels frozen on 1 Nov (`provenance/glimmer-judge-lora-v6/labels-frozen.json`), the Kaggle judge
beats Muse Spark Open's saved replies on the rows where exactly one of the two is right (exact two-sided sign test,
p < 0.05). Reported either way: rows right, rows with the same verdict as the Fireworks deployment, the bench and the
gate reads. If the wall cap stops the replay early, the rows replayed (a prefix in holdout order) are scored and the
count is stated. A failed conversion or a template that does not match training stops the run, reported as not run.
No new training and no new labels: the adapter's files are fixed by sha256 below.

**Inputs (Add Input):** `crossing-watch-code` (a `git archive` zip of one commit, carrying `GIT_SHA`) and the private
`cw-judge-lora-v6`: the adapter files exactly as `firectl model download` gives them (`adapter_config.json`,
`adapter_model-00001-of-00001.safetensors`, its index, the checkpoint metadata), plus `v6-holdout-prompts.jsonl` and
`gate-judge-prompt.json`. **Settings → Accelerator: GPU T4 x2. Internet: on.**

Weights trained in BF16 run here over a Q4_K_M base: expect a small drift from the served numbers.

In [ ]:
import json, subprocess, time
t0 = time.time()  # the provenance file gives each step's end in seconds from here
NVIDIA_SMI = subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=True).stdout
GPUS = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True,
                      check=True).stdout.splitlines()
HOST = f"Kaggle {len(GPUS)} x {GPUS[0].removeprefix('Tesla ')}"
print(NVIDIA_SMI, HOST)

## 1. Build llama.cpp's server for the T4 (about 28 minutes), and fix the LoRA converter

In [ ]:
%%bash
set -eo pipefail
cd /kaggle/working
[ -d llama.cpp ] || git clone --depth 1 --branch b11976 https://github.com/ggml-org/llama.cpp
cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=ON -DGGML_CUDA_NO_VMM=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DLLAMA_CURL=OFF | tail -3
cmake --build llama.cpp/build --config Release -j 4 --target llama-server 2>&1 | tail -2
# the fix: conversion/muse_glimmer.py unpermutes q/k with tensor.ndim, which LoraTorchTensor lacks (b11976 and master)
F=llama.cpp/convert_lora_to_gguf.py
grep -q 'def ndim' $F || sed -i 's/^    def dim(self) -> int:$/    @property\n    def ndim(self) -> int:\n        return len(self.shape)\n\n&/' $F
[ "$(grep -c 'def ndim' $F)" = 1 ] && python3 -c "import ast, sys; ast.parse(open(sys.argv[1]).read())" $F && echo 'ndim fix applied'

## 2. The base (pinned, sha256 checked), its config, and the v6 adapter (sha256, rank, shapes against the base)

In [ ]:
import glob, hashlib, math, os, shutil, struct
from huggingface_hub import hf_hub_download
marks = {'built': round(time.time() - t0)}
REPO, REVISION = 'meta-models/Muse-Glimmer-30B-GGUF', 'b44f1072cf75e918a1b82a3a17b45c4389b076e7'
BASE_GGUF = 'Muse-Glimmer-30B-KQuant-17GB-Q4_K_M.gguf'
BASE_SHA256 = '4cc57c0f51040a226e5a72cc47b7613f7772950e460a665f7083de89f183f60e'
CONFIG_REPO, CONFIG_REVISION = 'meta-models/Muse-Glimmer-30B', 'cec94fcfd6512f2cfe708138c32533b346ea4f69'


def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 24), b''):
            digest.update(block)
    return digest.hexdigest()


model = hf_hub_download(REPO, BASE_GGUF, revision=REVISION, local_dir='/tmp/glimmer')
assert sha256(model) == BASE_SHA256, 'the base is not the file Hugging Face published'
BASE_DIR = '/kaggle/working/base-config'  # config.json only: the converter needs no weights
hf_hub_download(CONFIG_REPO, 'config.json', revision=CONFIG_REVISION, local_dir=BASE_DIR)

src = os.path.dirname(glob.glob('/kaggle/input/**/pack.py', recursive=True)[0])
CW = '/kaggle/working/cw'
shutil.copytree(src, CW, dirs_exist_ok=True)

# the v6 adapter as `firectl model download` gave it on 4 Nov 2026, fixed here before the run
ADAPTER_FILES_SHA256 = {
    'adapter_config.json': 'b4ea83e72337afda1793d60eb78960fc864663faa4f0f5fa37756df8396b2ef1',
    'adapter_model-00001-of-00001.safetensors': '1a1f9c6ab8159141591b73d6b35ee257656843f9b3f13a9e43ce89bea045af1e',
}
TRAINED_RANK = 32  # glimmer-ft v6 train.sh: --lora-rank 32 --epochs 2 --learning-rate 1e-4
found = glob.glob('/kaggle/input/**/adapter_model-00001-of-00001.safetensors', recursive=True)
if len(found) != 1:
    raise RuntimeError(f'add the private cw-judge-lora-v6 dataset as an input (shards found: {found})')
INPUT = os.path.dirname(found[0])
for name, digest in ADAPTER_FILES_SHA256.items():
    assert sha256(f'{INPUT}/{name}') == digest, f'{name} is not the downloaded v6 adapter'
LORA = '/kaggle/working/lora'  # the converter reads adapter_config.json and adapter_model.safetensors from one folder
os.makedirs(LORA, exist_ok=True)
shutil.copy(f'{INPUT}/adapter_config.json', LORA)
if not os.path.exists(f'{LORA}/adapter_model.safetensors'):  # one shard holds the whole adapter
    os.symlink(found[0], f'{LORA}/adapter_model.safetensors')
adapter_config = json.load(open(f'{LORA}/adapter_config.json'))
assert adapter_config['r'] == TRAINED_RANK, adapter_config

with open(found[0], 'rb') as f:  # safetensors: 8-byte length, then a JSON header
    header = json.loads(f.read(struct.unpack('<Q', f.read(8))[0]))
tensors = {k: v['shape'] for k, v in header.items() if k != '__metadata__'}
base_config = json.load(open(f'{BASE_DIR}/config.json'))
text_config = base_config.get('text_config', base_config)
layers = {int(k.split('layers.')[1].split('.')[0]) for k in tensors if 'layers.' in k}
q_lora_a = next(v for k, v in tensors.items() if k.endswith('layers.0.self_attn.q_proj.lora_A.weight'))
# trained on this base: one LoRA per layer, and q_proj's A reads the base's hidden size
assert len(layers) == text_config['num_hidden_layers'], (len(layers), text_config['num_hidden_layers'])
assert q_lora_a == [TRAINED_RANK, text_config['hidden_size']], (q_lora_a, text_config['hidden_size'])
ADAPTER = {'files_sha256': ADAPTER_FILES_SHA256,
           'fw_checkpoint': json.load(open(f'{INPUT}/fw_checkpoint_metadata.json')),
           'r': adapter_config['r'], 'lora_alpha': adapter_config['lora_alpha'],
           'target_modules': adapter_config.get('target_modules'), 'tensors': len(tensors),
           'params': sum(math.prod(s) for s in tensors.values()), 'layers': len(layers),
           'base_architectures': base_config.get('architectures'),
           'lm_head_trained': any('lm_head' in k for k in tensors),
           'first_tensors': dict(list(tensors.items())[:4])}
marks['downloaded'] = round(time.time() - t0)
print(json.dumps(ADAPTER, indent=1))

## 3. Convert the adapter to GGUF (f16)

In [ ]:
LORA_GGUF = '/kaggle/working/glimmer-judge-lora-f16.gguf'
conv = subprocess.run(['python3', '/kaggle/working/llama.cpp/convert_lora_to_gguf.py', LORA, '--base', BASE_DIR,
                       '--outtype', 'f16', '--outfile', LORA_GGUF], capture_output=True, text=True)
print(conv.stderr[-3000:])
if conv.returncode or not os.path.exists(LORA_GGUF):
    raise RuntimeError('convert_lora_to_gguf.py failed (output above)')
ADAPTER['gguf'] = {'sha256': sha256(LORA_GGUF), 'bytes': os.path.getsize(LORA_GGUF)}
marks['converted'] = round(time.time() - t0)
ADAPTER['gguf']

## 4. Start the server with `--lora` and the training template, and check the rendered prompt

Training rendered every row with the cookbook's Glimmer renderer: `Reasoning strength: high.` and the training day as
the current date (v6 was launched Sun 1 Nov 2026, 18:21 IST: `2026-11-01`). `/apply-template` must give the training
header for the same message (the renderer's unit-test message, with v6's date), or nothing is asked.

In [ ]:
import urllib.request
LOG = '/kaggle/working/server.log'
TEMPLATE_KWARGS = {'reasoning_strength': 'high', 'current_date': '2026-11-01'}  # as the training rows were rendered
ARGS = ['-m', model, '--lora', LORA_GGUF, '-ngl', '99', '--split-mode', 'layer', '--jinja', '--port', '8080',
        '-np', '4', '-c', '16384',  # 4 slots of 4,096: a v6 prompt is at most 3,072 tokens with its reply
        '--chat-template-kwargs', json.dumps(TEMPLATE_KWARGS),
        '--temp', '0']  # the starter's evaluate() sampled at temperature 0.0
server = subprocess.Popen(['/kaggle/working/llama.cpp/build/bin/llama-server', *ARGS], stdout=open(LOG, 'a'),
                          stderr=subprocess.STDOUT)
for _ in range(120):
    if server.poll() is not None:
        raise RuntimeError('llama-server exited:\n' + open(LOG).read()[-3000:])
    try:
        if urllib.request.urlopen('http://127.0.0.1:8080/health', timeout=5).status == 200:
            break
    except OSError:
        time.sleep(10)
else:
    raise RuntimeError('llama-server never got healthy:\n' + open(LOG).read()[-3000:])
PROPS = json.load(urllib.request.urlopen('http://127.0.0.1:8080/props', timeout=30))
LORAS = json.load(urllib.request.urlopen('http://127.0.0.1:8080/lora-adapters', timeout=30))
marks['server_ready'] = round(time.time() - t0)


def post(path, body):
    req = urllib.request.Request(f'http://127.0.0.1:8080{path}', json.dumps(body).encode(),
                                 {'Content-Type': 'application/json'})
    return json.load(urllib.request.urlopen(req, timeout=60))


EXPECTED = ('<|start|>system<|message|>You are a helpful AI assistant.\nKnowledge cutoff: 2026-01-04.\n'
            'Current date: 2026-11-01.\n\nReasoning strength: high.\n\n# Valid recipients: "self", "user".<|eot|>'
            '<|start|>user<|message|> hi <|eot|><|start|>assistant')
rendered = post('/apply-template', {'messages': [{'role': 'user', 'content': ' hi '}]})['prompt']
TEMPLATE_MATCH = rendered.removeprefix('<|begin_of_text|>') == EXPECTED
print('build:', PROPS.get('build_info'), '| lora:', LORAS, '| template matches training:', TEMPLATE_MATCH)
if not TEMPLATE_MATCH or not LORAS:
    raise RuntimeError(f'not the trained setup: lora {LORAS}, rendered {rendered!r}')

## 5. The public bench, through the product's own judge path, one vote a letter

Only the tuned Glimmer is asked (`JUDGE_MODEL_URL` is this server, the Meta key is left out); the bench's Spark column is kept as committed for comparison.

In [ ]:
BENCH_OUT = '/kaggle/working/glimmer-judge-lora-bench.txt'
BENCH_JSON = '/kaggle/working/glimmer-judge-lora-bench.json'
bench_items = json.load(open(f'{CW}/kaggle/judge-bench-public.json'))
# any Glimmer verdict already in the file is base Glimmer's: drop it, or judge_bench.py would ask the tuned one nothing
bench_items = [{k: v for k, v in i.items() if k not in ('glimmer', 'glimmer_votes')} for i in bench_items]
json.dump(bench_items, open(BENCH_JSON, 'w'), indent=1, ensure_ascii=False)
bench_env = {**{k: v for k, v in os.environ.items() if k != 'MODEL_API_KEY'},
             'JUDGE_MODEL_URL': 'http://127.0.0.1:8080/v1', 'JUDGE_VOTES': '1'}
started = time.time()
bench = subprocess.run(['python3', 'judge_bench.py', '--out', BENCH_JSON, '--glimmer', '1'], cwd=CW, env=bench_env,
                       capture_output=True, text=True)
bench_s = round(time.time() - started)
replies = bench.stderr.count('open model reply:')
git_sha = open(f'{CW}/GIT_SHA').read().strip() if os.path.exists(f'{CW}/GIT_SHA') else None
with open(BENCH_OUT, 'w') as f:
    f.write(f'judge_bench.py: exit {bench.returncode}, {bench_s} s on {HOST}, {replies} replies from the tuned Glimmer '
            f'(LoRA {ADAPTER["gguf"]["sha256"][:12]}), JUDGE_VOTES=1, temp 0, git {git_sha}\n\n'
            f'{bench.stdout}\n{bench.stderr[-3000:]}')
print(open(BENCH_OUT).read())
marks['benched'] = round(time.time() - t0)

judged = json.load(open(BENCH_JSON))
flagged = lambda planted: sum(any(v for v in i.get('glimmer', []) if v) for i in judged if i['planted'] == planted)
BENCH_SUMMARY = {'file': 'kaggle/judge-bench-public.json', 'exit': bench.returncode, 'seconds': bench_s,
                 'replies': replies, 's_per_reply': round(bench_s / replies, 1) if replies else None,
                 'lies_flagged': f"{flagged(True)}/{sum(i['planted'] for i in judged)}",
                 'clean_flagged': f"{flagged(False)}/{sum(not i['planted'] for i in judged)}",
                 'unreadable': sum(v is None for i in judged for v in i.get('glimmer', []))}
print(json.dumps(BENCH_SUMMARY, indent=1))

## 6. Our real gate letter, read 6 times on each prompt (3 plain, 3 with the product's JSON schema)

In [ ]:
def chat(body, timeout=900):
    req = urllib.request.Request('http://127.0.0.1:8080/v1/chat/completions', json.dumps(body).encode(),
                                 {'Content-Type': 'application/json'})
    return json.load(urllib.request.urlopen(req, timeout=timeout))


def problems(text):
    """The judge's list, or None if the reply is not the JSON it was asked for."""
    try:
        found = json.loads(text)['problems']
        return found if isinstance(found, list) else None
    except (ValueError, KeyError, TypeError):
        return None


GATE = json.load(open(glob.glob('/kaggle/input/**/gate-judge-prompt.json', recursive=True)[0]))
GATE_READS = {}
for which in ('served_3nov', 'product_today'):
    prompt = GATE[which]['prompt']
    assert hashlib.sha256(prompt.encode()).hexdigest() == GATE[which]['prompt_sha256'], which
    reads = []
    for schema in (False, False, False, True, True, True):  # as judge_gate.py asked the Fireworks deployment
        body = {'temperature': 0, 'max_tokens': 400, 'messages': [{'role': 'user', 'content': prompt}]}
        if schema:
            body['response_format'] = {'type': 'json_schema',
                                       'json_schema': {'name': 'problems', 'schema': GATE['schema']}}
        started = time.time()
        r = chat(body)
        text = r['choices'][0]['message'].get('content') or ''
        reads.append({'schema': schema, 'reply': text, 'problems': problems(text),
                      'finish': r['choices'][0].get('finish_reason'), 'usage': r.get('usage'),
                      'wall_s': round(time.time() - started, 2)})
    GATE_READS[which] = {'prompt_sha256': GATE[which]['prompt_sha256'], 'built_with': GATE[which]['built_with'],
                         'reads': reads}
    print(which, [read['problems'] for read in reads])
GATE_RECORD = {'letter_sha256': GATE['letter_sha256'], 'host': HOST, 'lora_gguf_sha256': ADAPTER['gguf']['sha256'],
               **GATE_READS}
with open('/kaggle/working/gate-letter-judge-kaggle.json', 'w') as f:
    json.dump(GATE_RECORD, f, indent=1, ensure_ascii=False)
marks['gate_read'] = round(time.time() - t0)

## 7. All 1,545 v6 holdout prompts, as the Fireworks deployment was asked them

Scored after the run, outside this notebook, against the labels frozen on 1 Nov and the saved Fireworks and Muse Spark Open replies (`provenance/glimmer-judge-lora-v6/`).

In [ ]:
import threading
from concurrent.futures import ThreadPoolExecutor
PROMPTS = [json.loads(line) for line in open(glob.glob('/kaggle/input/**/v6-holdout-prompts.jsonl', recursive=True)[0])]
assert [p['index'] for p in PROMPTS] == list(range(1545)), 'not the 1,545 v6 holdout prompts in holdout order'
REPLAY = '/kaggle/working/predictions_kaggle.jsonl'
WALL_CAP_S = 7 * 3600  # Kaggle stops a session at 12 h; nothing new is asked after this, and what is done is kept
WORKERS = 4            # one per server slot (-np 4)
lock = threading.Lock()
replay_started = time.time()


def ask(p):
    if time.time() - replay_started > WALL_CAP_S:
        return None
    started = time.time()
    try:
        r = chat({'temperature': 0, 'max_tokens': 256, 'messages': [{'role': 'user', 'content': p['content']}]})
        line = {'index': p['index'], 'id': p['id'], 'reply': r['choices'][0]['message'].get('content') or '',
                'finish': r['choices'][0].get('finish_reason'), 'usage': r.get('usage'),
                'wall_s': round(time.time() - started, 2)}
    except Exception as e:  # kept as a failed row, never dropped
        line = {'index': p['index'], 'id': p['id'], 'error': f'{type(e).__name__}: {str(e)[:300]}',
                'wall_s': round(time.time() - started, 2)}
    with lock, open(REPLAY, 'a') as f:
        f.write(json.dumps(line, ensure_ascii=False) + '\n')
        done = sum(1 for _ in open(REPLAY))
    if done % 100 == 0:
        print(f'{done} of {len(PROMPTS)} in {time.time() - replay_started:.0f} s', flush=True)
    return line


open(REPLAY, 'w').close()
with ThreadPoolExecutor(WORKERS) as pool:
    replayed = [r for r in pool.map(ask, PROMPTS) if r]
replay_s = round(time.time() - replay_started)
marks['replayed'] = round(time.time() - t0)
answered = [r for r in replayed if 'error' not in r]
REPLAY_SUMMARY = {
    'asked': len(replayed), 'of': len(PROMPTS), 'answered': len(answered), 'errors': len(replayed) - len(answered),
    'stopped_by_wall_cap': len(replayed) < len(PROMPTS), 'seconds': replay_s,
    'finish_stop': sum(r['finish'] == 'stop' for r in answered),
    'readable': sum(problems(r['reply']) is not None for r in answered),
    'flagged': sum(bool(problems(r['reply'])) for r in answered),
    'prompt_tokens': sum((r['usage'] or {}).get('prompt_tokens', 0) for r in answered),
    'completion_tokens': sum((r['usage'] or {}).get('completion_tokens', 0) for r in answered),
    'median_wall_s': sorted(r['wall_s'] for r in answered)[len(answered) // 2] if answered else None}
print(json.dumps(REPLAY_SUMMARY, indent=1))

## 8. The run record: `glimmer-judge-lora-provenance.json`

Download it with `gate-letter-judge-kaggle.json`, `predictions_kaggle.jsonl`, the bench files and the converted adapter (`glimmer-judge-lora-f16.gguf`).

In [ ]:
provenance = {
    'notebook': 'kaggle/glimmer-judge-lora.ipynb',
    'team': "Run by the author on the author's own Kaggle account (free 2 x T4).",
    'promoted_model': 'accounts/<fireworks-account>/models/muse-starter-sl-judge-v6-main',
    'training_session': 'ts-17bf2de281bb4cbe8424de262befb01b',
    'code': {'git_sha': git_sha},
    'nvidia_smi': NVIDIA_SMI,
    'adapter': ADAPTER,
    'base': {'repo': REPO, 'revision': REVISION, 'file': BASE_GGUF, 'sha256_checked': BASE_SHA256,
             'config': f'{CONFIG_REPO}@{CONFIG_REVISION}'},
    'llama_cpp': {'build_info': PROPS.get('build_info'), 'args': ARGS, 'lora_adapters': LORAS,
                  'converter_fix': 'LoraTorchTensor.ndim added (sed, cell 3)'},
    'template': {'kwargs': TEMPLATE_KWARGS, 'matches_training_renderer': TEMPLATE_MATCH},
    'bench': BENCH_SUMMARY,
    'gate_letter': {k: [read['problems'] for read in v['reads']] for k, v in GATE_READS.items()},
    'replay': REPLAY_SUMMARY,
    'elapsed_s': marks,
}
with open('/kaggle/working/glimmer-judge-lora-provenance.json', 'w') as f:
    json.dump(provenance, f, indent=1, ensure_ascii=False)
print(json.dumps({k: provenance[k] for k in ('bench', 'gate_letter', 'replay', 'elapsed_s')}, indent=1))
# keep the outputs small: the build, the code copy and the adapter link go; the converted adapter stays
for path in ('/kaggle/working/llama.cpp', '/kaggle/working/cw', '/kaggle/working/lora', '/kaggle/working/base-config'):
    shutil.rmtree(path, ignore_errors=True)